# Pulling data from APIs
Fun fact: Planet Terp has its own API, and it is completely public! We're going to be pulling data from that API. 

To see information about each endpoint and its parameters, visit https://planetterp.com/api/. 

The /courses endpoint gives us all of our courses, in alphabetical order.
Try going to https://planetterp.com/api/v1/courses. What do you see?

In [10]:
import requests

# The base URL for every PlanetTerp API call
BASE_URL = 'https://planetterp.com/api/v1'

# Make a GET request to the /courses endpoint
response = requests.get(f'{BASE_URL}/courses')

# The status code tells you if the request worked
# 200 = success, 404 = not found, 403 = forbidden, 500 = server error
print(f"Status code: {response.status_code}")

Status code: 200


In [11]:
# .json() converts the raw response text into a Python list of dictionaries
courses = response.json()

print(f"Number of courses returned: {len(courses)}")
print()

# Let's peek at the first course to understand the data structure
print("First course:")
print(courses[0])

Number of courses returned: 100

First course:
{'average_gpa': 3.7170258620689656, 'professors': ['Jordan Boyd-Graber', 'A Seyed', 'Vanessa Frias-Martinez', 'Geoffrey Malafsky', 'Vanessa Frias-Martinez', 'Naeemul Hassan', 'David Patrick', 'Vanessa Frias-Martinez', 'David Patrick', 'Vanessa Frias-Martinez', 'Vanessa Frias-Martinez', 'Vanessa Frias-Martinez', 'Naeemul Hassan', 'Naeemul Hassan', 'Zach Drake', 'Naeemul Hassan', 'Brian Gillikin'], 'department': 'INST', 'course_number': '737', 'name': 'INST737', 'title': 'Introduction to Data Science', 'credits': 3, 'description': '<b>Prerequisite:</b> INST627 and (LBSC690, LBSC671, INFM603, or JOUR652).\n<b>Restriction:</b> Permission of INFO-College of Information Studies.\nAn exploration of some of the best and most general approaches to get the most information out of data through clustering, classification, and regression techniques.', 'is_recent': True, 'geneds': None}


### Using query parameters

Most APIs let you filter or shape what gets returned using **query parameters** — the `?key=value` parts you've seen in URLs.

The PlanetTerp `/courses` endpoint supports:
* `name` Filters to courses whose name starts with this string (e.g. `JOUR`) |
* `reviews` set to `true` to include student reviews 
* `limit` sets the max number of results to return (default: 100) 
* `offset` skip the first N results , which is useful for paging through large datasets 

Pass them as a dictionary to the `params` argument of `requests.get()`.

In [12]:
# Pull only JOUR courses, limit to 10 results
response = requests.get(
    f'{BASE_URL}/courses',
    params={
        'department': 'JOUR',
        'limit': 10
    }
)

jour_courses = response.json()
print(f"JOUR courses returned: {len(jour_courses)}")

# Print name, title, and average GPA for each
for course in jour_courses:
    gpa = course['average_gpa']
    gpa_str = f"{gpa:.2f}" if gpa else "N/A"
    print(f"{course['name']}: {course['title']} | Avg GPA: {gpa_str}")

JOUR courses returned: 10
JOUR199: Survey Apprenticeship | Avg GPA: N/A
JOUR396: Supervised Internship | Avg GPA: 3.73
JOUR399: Supervised Internship | Avg GPA: 3.64
JOUR620: Public Affairs Reporting | Avg GPA: 3.36
JOUR352: Interactive Design and Development | Avg GPA: 3.40
JOUR289E: Media Law and Ethics in the Digital Age | Avg GPA: 3.50
JOUR453: News Coverage of Racial Issues | Avg GPA: 3.55
JOUR262: News Videography | Avg GPA: 3.43
JOUR368V: Topics in Broadcast and Electronic Media; Advanced Video Storytelling | Avg GPA: 3.89
JOUR603: News Videography | Avg GPA: 3.43


### Loading API results into a dataframe

Once you have a list of dictionaries from the API, you can drop it into pandas for analysis.

In [13]:
import pandas as pd

# Convert the list of course dictionaries directly into a DataFrame
df = pd.DataFrame(jour_courses)

# Drop columns we don't need for this analysis
df = df[['name', 'title', 'credits', 'average_gpa', 'professors']]

df

,name,title,credits,average_gpa,professors
0,JOUR199,Survey Apprenticeship,1,0.000000,"[Christine Harvey, Adrianne Flynn, Karen Denny..."
1,JOUR396,Supervised Internship,2,3.734298,"[Adrianne Flynn, Karen Denny, Karen Denny, Kar..."
2,JOUR399,Supervised Internship,1,3.640000,"[Christine Harvey, Adrianne Flynn]"
3,JOUR620,Public Affairs Reporting,3,3.356828,"[Carl Stepp, Adrianne Flynn, Elaine Povich, Sa..."
4,JOUR352,Interactive Design and Development,3,3.401651,"[Alexander Pyles, Leslie Walker, Mark Smith, C..."
5,JOUR289E,Media Law and Ethics in the Digital Age,3,3.496862,"[Deborah Nelson, Sandra Banisky]"
6,JOUR453,News Coverage of Racial Issues,3,3.549186,"[Anne Rosen, Kimberly Davis, Anne Rosen, Anne ..."
7,JOUR262,News Videography,3,3.428494,"[Bethany Swain, Charles MacDonald, Kevin Johns..."
8,JOUR368V,Topics in Broadcast and Electronic Media; Adva...,3,3.887755,[Bethany Swain]
9,JOUR603,News Videography,3,3.425000,"[Kimberlee Shaffir, Bethany Swain, Luke Rollin..."


In [14]:
# Which JOUR courses have the highest average GPA?
df.sort_values('average_gpa', ascending=False).dropna(subset=['average_gpa'])

,name,title,credits,average_gpa,professors
8,JOUR368V,Topics in Broadcast and Electronic Media; Adva...,3,3.887755,[Bethany Swain]
1,JOUR396,Supervised Internship,2,3.734298,"[Adrianne Flynn, Karen Denny, Karen Denny, Kar..."
2,JOUR399,Supervised Internship,1,3.640000,"[Christine Harvey, Adrianne Flynn]"
6,JOUR453,News Coverage of Racial Issues,3,3.549186,"[Anne Rosen, Kimberly Davis, Anne Rosen, Anne ..."
5,JOUR289E,Media Law and Ethics in the Digital Age,3,3.496862,"[Deborah Nelson, Sandra Banisky]"
7,JOUR262,News Videography,3,3.428494,"[Bethany Swain, Charles MacDonald, Kevin Johns..."
9,JOUR603,News Videography,3,3.425000,"[Kimberlee Shaffir, Bethany Swain, Luke Rollin..."
4,JOUR352,Interactive Design and Development,3,3.401651,"[Alexander Pyles, Leslie Walker, Mark Smith, C..."
3,JOUR620,Public Affairs Reporting,3,3.356828,"[Carl Stepp, Adrianne Flynn, Elaine Povich, Sa..."
0,JOUR199,Survey Apprenticeship,1,0.000000,"[Christine Harvey, Adrianne Flynn, Karen Denny..."


## Using API keys

Most real-world APIs require an **API key** — a unique token that identifies you and lets the API track your usage. The Census Bureau API is free, but you need to register for a key.

**Get your key here:** https://api.census.gov/data/key_signup.html 

### Where does the key go?

With the Census API, the key is passed as a **query parameter** called `key` — the same way you passed `department` or `limit` to PlanetTerp:

```python
params = {
    'get': 'NAME,B01001_001E',
    'for': 'state:*',
    'key': 'YOUR_KEY_HERE'   # <-- key goes here
}
```

**Never hardcode your API key in a notebook you share or push to GitHub.** Store it in a separate file and read it in.

To store your key, create a new file in the root of your directory with the name `census_key.txt`. Then, navigate to your `.gitignore` file and add the line `census_key.txt` so you do NOT push it to GitHub.

In [15]:
# Store your key in a plain text file called census_key.txt (one line, just the key)
# That file should be in your .gitignore so it never gets pushed

import os

key_path = ('../census_key.txt')

if os.path.exists(key_path):
    with open(key_path) as f:
        CENSUS_KEY = f.read().strip()
    print("Key loaded.")
else:
    CENSUS_KEY = None
    print("No key file found.")

Key loaded.


### A note on the Census API response format

The Census API returns data differently than PlanetTerp. It returns an **array of arrays** — like a spreadsheet where the first row is column headers:

```
[
  ["NAME", "B01001_001E", "B19013_001E", "state"],   <- headers
  ["Alabama", "5054253", "62027", "01"],
  ["Alaska",  "733971",  "89336", "02"],
  ...
]
```

This means you can't call `.json()` and drop it straight into a DataFrame — you need to split the header row from the data rows first.

In [16]:
import requests
import pandas as pd

# We'll pull total population and median household income for every state
# Variable codes: https://api.census.gov/data/2023/acs/acs5/variables.html
#   B01001_001E = total population
#   B19013_001E = median household income

params = {
    'get': 'NAME,B01001_001E,B19013_001E',
    'for': 'state:*',          # all 50 states + DC + Puerto Rico
}

if CENSUS_KEY:
    params['key'] = CENSUS_KEY

response = requests.get('https://api.census.gov/data/2023/acs/acs5', params=params)
print(f"Status: {response.status_code}")

# The response is an array of arrays — peek at the first two rows
data = response.json()
print("Headers:", data[0])
print("First row:", data[1])

Status: 200
Headers: ['NAME', 'B01001_001E', 'B19013_001E', 'state']
First row: ['Alabama', '5054253', '62027', '01']


In [ ]:
# Split headers from data rows and load into a df
headers = data[0]
rows = data[1:]

df_census = pd.DataFrame(rows, columns=headers)

# Rename columns to something readable
df_census = df_census.rename(columns={
    'NAME': 'state',
    'B01001_001E': 'population',
    'B19013_001E': 'median_household_income'
})

# Census API returns all values as strings — convert numbers
df_census['population'] = pd.to_numeric(df_census['population'])
df_census['median_household_income'] = pd.to_numeric(df_census['median_household_income'])

df_census.head(10)

,state,population,median_household_income,state
0,Alabama,5054253,62027,01
1,Alaska,733971,89336,02
2,Arizona,7268175,76872,04
3,Arkansas,3032651,58773,05
4,California,39242785,96334,06
5,Colorado,5810774,92470,08
6,Connecticut,3598348,93760,09
7,Delaware,1005872,82855,10
8,District of Columbia,672079,106287,11
9,Florida,21928881,71711,12


In [18]:
# Which states have the highest median household income?
df_census.sort_values('median_household_income', ascending=False).head(10)[
    ['state', 'population', 'median_household_income']
]

,state,state,population,median_household_income
8,District of Columbia,11,672079,106287
20,Maryland,24,6170738,101652
21,Massachusetts,25,6992395,101341
30,New Jersey,34,9267014,101050
11,Hawaii,15,1445635,98317
4,California,06,39242785,96334
29,New Hampshire,33,1387834,95628
47,Washington,53,7740984,94952
6,Connecticut,09,3598348,93760
5,Colorado,08,5810774,92470


## Using OAuth

OAuth is a bit more complex compared to a regular API key. Instead of just passing a key with each request, you first have to **exchange your credentials for a temporary token**, then use that token to make requests.

1. Register your app at developer.spotify.com → get CLIENT_ID and CLIENT_SECRET
2. Send CLIENT_ID + CLIENT_SECRET to Spotify's token endpoint
3. Spotify gives back an access_token (valid for 1 hour)
4. Pass that token in the Authorization header of every API request

This gives you access to public Spotify data without needing a user to log in.

**To get your credentials:**
1. Go to https://developer.spotify.com/dashboard and log in
2. Click "Create App"
3. Copy your **Client ID** and **Client Secret**

### Storing credentials safely

Never paste your Client ID or Secret directly in the notebook. Store them in a file called `spotify_creds.txt` with two lines:

```
your_client_id_here
your_client_secret_here
```

Add `spotify_creds.txt` to your `.gitignore` so it never gets pushed.

In [ ]:
import os

creds_path = os.path.join(os.getcwd(), 'spotify_creds.txt')

if os.path.exists(creds_path):
    with open(creds_path) as f:
        lines = f.read().strip().splitlines()
        CLIENT_ID = lines[0]
        CLIENT_SECRET = lines[1]
    print("Credentials loaded.")
else:
    CLIENT_ID = CLIENT_SECRET = None
    print(f"No credentials file found. Create spotify_creds.txt in: {os.getcwd()}")

### Exchange credentials for a token

Spotify needs CLIENT_ID and CLIENT_SECRET to be **Base64 encoded** and sent as a POST request. This is different from everything we've done so far — instead of a GET request with params, we're sending a POST with form data and a special header.

In [ ]:
import requests
import base64

# Base64 encode "CLIENT_ID:CLIENT_SECRET"
credentials = base64.b64encode(f"{CLIENT_ID}:{CLIENT_SECRET}".encode()).decode()

# POST to Spotify's token endpoint
response = requests.post(
    'https://accounts.spotify.com/api/token',
    headers={
        'Authorization': f'Basic {credentials}',
        'Content-Type': 'application/x-www-form-urlencoded'
    },
    data={'grant_type': 'client_credentials'}
)

token_data = response.json()
ACCESS_TOKEN = token_data['access_token']
print(f"Token received! Expires in: {token_data['expires_in']} seconds ({token_data['expires_in']//60} minutes)")

### Step 2: Use the token in the Authorization header

With the Census API, the key went in the query params (`?key=...`). With OAuth, the token goes in the **request header** as a Bearer token:

```python
headers = {'Authorization': f'Bearer {ACCESS_TOKEN}'}
```

Every API request from here on needs this header.

In [ ]:
# Build the auth header — reuse this for every Spotify request
headers = {'Authorization': f'Bearer {ACCESS_TOKEN}'}

# Search for an artist
response = requests.get(
    'https://api.spotify.com/v1/search',
    headers=headers,
    params={
        'q': 'Kendrick Lamar',
        'type': 'artist',
        'limit': 5
    }
)

results = response.json()
for artist in results['artists']['items']:
    print(f"{artist['name']}")
    print(f"  Followers: {artist['followers']['total']:,} | Popularity: {artist['popularity']}/100")
    print(f"  Genres: {', '.join(artist['genres'][:3]) if artist['genres'] else 'N/A'}")
    print()

In [ ]:
import pandas as pd

# Pull the top 10 tracks for the first result
artist_id = results['artists']['items'][0]['id']
artist_name = results['artists']['items'][0]['name']

response = requests.get(
    f'https://api.spotify.com/v1/artists/{artist_id}/top-tracks',
    headers=headers,
    params={'market': 'US'}
)

tracks = response.json()['tracks']

df_tracks = pd.DataFrame([{
    'track': t['name'],
    'album': t['album']['name'],
    'popularity': t['popularity'],
    'duration_sec': t['duration_ms'] // 1000
} for t in tracks])

print(f"Top tracks for {artist_name}:")
df_tracks.sort_values('popularity', ascending=False)